# LLM Serving Benchmark Experiment Plan

This notebook documents the experiment design, expected phenomena, execution commands, and result analysis for the vLLM / SGLang serving benchmark project.

Main focus:

1. Throughput vs concurrency
2. TTFT under short and long prompt workloads
3. P95 latency under high concurrency
4. Prefix cache / RadixAttention behavior
5. KV cache allocation and memory utilization

## 1. Hardware Assumption

Target cloud GPU:

- RTX 4090 24GB
- Single GPU
- Model: Qwen/Qwen2.5-7B-Instruct
- Initial context length: 8192 if stable, otherwise 4096

The goal is not only to run the model, but to stress different phases of LLM serving:

- Prefill phase: dominated by prompt length
- Decode phase: dominated by generated tokens and concurrent scheduling
- KV cache pressure: dominated by sequence length and number of active requests

## 2. Expected Phenomena

### Short prompt, high concurrency

Expected:

- Request throughput increases from low to medium concurrency.
- At high concurrency, throughput growth slows down or plateaus.
- P95 latency increases because requests spend more time waiting in the queue.

### Long prompt

Expected:

- TTFT is much higher than short prompt because prefill work is heavier.
- Throughput is lower because each request consumes more prefill compute and KV cache space.

### Shared prefix

Expected:

- With prefix cache / RadixAttention enabled, repeated shared system prompts should reduce redundant prefill.
- TTFT should become lower after cache warmup.
- Throughput may improve when many requests share the same prefix.

### Prefix cache off vs on

Expected:

- Prefix cache off: shared_prefix behaves similarly to long prompt.
- Prefix cache on: shared_prefix should show lower TTFT and possibly higher throughput.

## 3. Start vLLM Server

Run this in a separate terminal on the GPU machine.

In [ ]:
MODEL=Qwen/Qwen2.5-7B-Instruct \
PORT=8000 \
MAX_MODEL_LEN=8192 \
GPU_MEM_UTIL=0.92 \
MAX_NUM_SEQS=64 \
MAX_NUM_BATCHED_TOKENS=16384 \
PREFIX_CACHE=1 \
bash launch/launch_vllm.sh

If OOM happens, use the conservative configuration.

In [ ]:
MODEL=Qwen/Qwen2.5-7B-Instruct \
PORT=8000 \
MAX_MODEL_LEN=4096 \
GPU_MEM_UTIL=0.90 \
MAX_NUM_SEQS=32 \
MAX_NUM_BATCHED_TOKENS=8192 \
PREFIX_CACHE=1 \
bash launch/launch_vllm.sh

## 4. Run vLLM Full Sweep

In [ ]:
ENGINE=vllm \
BASE_URL=http://127.0.0.1:8000/v1 \
MODEL=Qwen/Qwen2.5-7B-Instruct \
bash scripts/run_cloud_sweep.sh

## 5. Start SGLang Server

Run this after stopping vLLM.

In [ ]:
MODEL=Qwen/Qwen2.5-7B-Instruct \
PORT=30000 \
MAX_MODEL_LEN=8192 \
MEM_FRACTION=0.90 \
MAX_RUNNING_REQUESTS=64 \
MAX_PREFILL_TOKENS=16384 \
RADIX_CACHE=1 \
bash launch/launch_sglang.sh

If OOM happens, use the conservative configuration.

In [ ]:
MODEL=Qwen/Qwen2.5-7B-Instruct \
PORT=30000 \
MAX_MODEL_LEN=4096 \
MEM_FRACTION=0.85 \
MAX_RUNNING_REQUESTS=32 \
MAX_PREFILL_TOKENS=8192 \
RADIX_CACHE=1 \
bash launch/launch_sglang.sh

## 6. Run SGLang Full Sweep

In [ ]:
ENGINE=sglang \
BASE_URL=http://127.0.0.1:30000/v1 \
MODEL=Qwen/Qwen2.5-7B-Instruct \
bash scripts/run_cloud_sweep.sh

## 7. Load Summary CSV

After running experiments, set the path to the generated summary file.

In [ ]:
import pandas as pd
from pathlib import Path

summary_path = Path("experiments/vllm_xxxxx/processed/summary.csv")
df = pd.read_csv(summary_path)
df

## 8. Basic Analysis

Check throughput and latency trends by workload and concurrency.

In [ ]:
cols = [
    "engine",
    "workload",
    "concurrency",
    "num_successful",
    "request_throughput_req_s",
    "output_throughput_tok_s",
    "e2e_latency_p95_s",
    "ttft_p95_s",
    "tpot_mean_s",
]

df[cols].sort_values(["workload", "concurrency"])

## 9. Interpretation Template

Use the following questions to write the final README analysis:

1. At which concurrency does throughput start to plateau?
2. Does P95 latency grow faster after a certain concurrency?
3. How much higher is long-prompt TTFT than short-prompt TTFT?
4. Does shared-prefix workload benefit from prefix cache / RadixAttention?
5. What is the tradeoff between throughput and latency?
6. What configuration is stable on RTX 4090 24GB?

## 10. Final Resume-Level Interpretation

A good final explanation should look like:

> I implemented a custom OpenAI-compatible asynchronous benchmark client and evaluated vLLM / SGLang under short-prompt, long-prompt, and shared-prefix workloads. The experiments show how concurrency improves batching efficiency at low-to-medium load, while high concurrency increases P95 latency due to queueing and KV cache pressure. Long prompts significantly increase TTFT due to heavier prefill, while shared-prefix workloads can benefit from prefix cache / RadixAttention by reducing redundant KV cache computation.